# Site-Safe PPE: reproducible YOLO11m training on Google Colab

This notebook trains the PPE detector with the fixed Site-Safe v5 configuration and evaluates the selected `best.pt` once on the held-out test split. The validation split is used for early stopping; the test split is not used for training or model selection.

Fixed training settings: YOLO11m, 150 epochs, 640-pixel images, batch size 16, patience 25, AdamW, and `lr0=0.001`. A fixed seed and deterministic-mode controls are also enabled. The notebook contains no assumed performance values: metrics are printed and saved only after the actual test evaluation runs.


## 1. Select a GPU runtime and install the pinned training dependency

In Colab, choose **Runtime > Change runtime type > T4 GPU** (or another CUDA GPU) before continuing. The training cell deliberately refuses to run without CUDA. Pinning Ultralytics to the repository's verified version makes the run easier to reproduce.


In [ ]:
%pip install -q "ultralytics==8.4.41" "PyYAML==6.0.3"


## 2. Put the dataset in Colab

Recommended: upload either the `ppe_dataset/` folder or a ZIP containing it to Google Drive, then edit `DATASET_SOURCE` below. The folder must contain `data.yaml` plus `train/`, `valid/`, and `test/`. The source is copied into Colab; Drive data is not modified.

For a one-off browser upload instead, run `from google.colab import files; files.upload()` in a temporary cell, upload the ZIP, set `MOUNT_GOOGLE_DRIVE = False`, and set `DATASET_SOURCE` to the uploaded path under `/content`. A Drive mount is usually more reliable for a dataset of this size.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import hashlib
import json
import shutil

MOUNT_GOOGLE_DRIVE = True
DATASET_SOURCE = Path("/content/drive/MyDrive/site-safe/ppe_dataset.zip")  # EDIT ME

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

WORKSPACE = Path("/content/site-safe")
AI_MODULE_ROOT = WORKSPACE / "ai-module"
DATASET_ROOT = AI_MODULE_ROOT / "ppe_dataset"
IMPORT_ROOT = Path("/content/sitesafe_dataset_import")

def looks_like_ppe_dataset(path: Path) -> bool:
    return (
        (path / "data.yaml").is_file()
        and all((path / split / "images").is_dir() for split in ("train", "valid", "test"))
        and all((path / split / "labels").is_dir() for split in ("train", "valid", "test"))
    )

def locate_ppe_dataset(search_root: Path) -> Path:
    if looks_like_ppe_dataset(search_root):
        return search_root
    candidates = sorted(
        {p.parent for p in search_root.rglob("data.yaml") if looks_like_ppe_dataset(p.parent)}
    )
    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one PPE dataset under {search_root}, found {len(candidates)}: {candidates}"
        )
    return candidates[0]

if not DATASET_SOURCE.exists():
    raise FileNotFoundError(f"Dataset source does not exist: {DATASET_SOURCE}")

if DATASET_SOURCE.is_file():
    if DATASET_SOURCE.suffix.lower() != ".zip":
        raise ValueError("DATASET_SOURCE must be a dataset directory or a .zip file")
    if IMPORT_ROOT.exists():
        shutil.rmtree(IMPORT_ROOT)
    IMPORT_ROOT.mkdir(parents=True)
    with ZipFile(DATASET_SOURCE) as archive:
        archive.extractall(IMPORT_ROOT)
    source_root = locate_ppe_dataset(IMPORT_ROOT)
else:
    source_root = locate_ppe_dataset(DATASET_SOURCE)

if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source_root, DATASET_ROOT)
DATA_YAML = DATASET_ROOT / "data.yaml"
print(f"Dataset copied to: {DATASET_ROOT}")
print(f"Dataset YAML:      {DATA_YAML}")

def label_inventory_sha256(dataset_root: Path) -> str:
    digest = hashlib.sha256()
    for label_path in sorted(dataset_root.glob("*/labels/*.txt")):
        relative = label_path.relative_to(dataset_root).as_posix()
        digest.update(relative.encode() + b"\0" + label_path.read_bytes() + b"\n")
    return digest.hexdigest()

def normalize_labels_for_detection(dataset_root: Path) -> dict:
    source_hash = label_inventory_sha256(dataset_root)
    summary = {}
    for split in ("train", "valid", "test"):
        converted_rows = 0
        files_with_polygons = 0
        mixed_format_files = 0
        for label_path in sorted((dataset_root / split / "labels").glob("*.txt")):
            original_rows = [line.strip() for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip()]
            row_lengths = [len(row.split()) for row in original_rows]
            has_boxes = any(length == 5 for length in row_lengths)
            has_polygons = any(length > 5 for length in row_lengths)
            if not has_polygons:
                continue
            files_with_polygons += 1
            mixed_format_files += int(has_boxes)
            normalized_rows = []
            for line_number, row in enumerate(original_rows, start=1):
                fields = row.split()
                try:
                    class_id = int(fields[0])
                    coordinates = [float(value) for value in fields[1:]]
                except (IndexError, ValueError) as exc:
                    raise ValueError(f"Cannot parse {label_path}:{line_number}") from exc
                if len(coordinates) == 4:
                    normalized_rows.append(row)
                    continue
                if len(coordinates) < 6 or len(coordinates) % 2:
                    raise ValueError(f"Invalid polygon row at {label_path}:{line_number}")
                xs = coordinates[0::2]
                ys = coordinates[1::2]
                x_min, x_max = min(xs), max(xs)
                y_min, y_max = min(ys), max(ys)
                box = ((x_min + x_max) / 2, (y_min + y_max) / 2, x_max - x_min, y_max - y_min)
                values = " ".join(f"{value:.10f}".rstrip("0").rstrip(".") or "0" for value in box)
                normalized_rows.append(f"{class_id} {values}")
                converted_rows += 1
            label_path.write_text("\n".join(normalized_rows) + "\n", encoding="utf-8")
        summary[split] = {
            "converted_polygon_rows": converted_rows,
            "files_with_polygons": files_with_polygons,
            "mixed_format_files": mixed_format_files,
        }
    return {
        "reason": "YOLO detection training requires one consistent box format; polygons were converted to their enclosing boxes in the temporary Colab copy",
        "source_labels_sha256": source_hash,
        "normalized_labels_sha256": label_inventory_sha256(dataset_root),
        "splits": summary,
    }

LABEL_NORMALIZATION = normalize_labels_for_detection(DATASET_ROOT)
print("\nDetection-label normalization applied to the temporary copy:")
print(json.dumps(LABEL_NORMALIZATION, indent=2))


## 3. Audit and fingerprint the uploaded dataset

The source dataset contains a small number of polygon rows mixed with box rows. The setup cell converts those polygons to enclosing boxes only in the temporary Colab copy, because mixed row shapes in one label file are unsafe for a detection run. This audit then counts images and annotation instances for every class and split, checks image/label pairing, and calculates a lightweight inventory fingerprint. The source and normalized label hashes and conversion counts are written into the run manifest. Training stops if labels are missing, orphaned, malformed, out of range, or use an unknown class.


In [ ]:
from collections import Counter
import hashlib
import json
import yaml

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
SPLIT_DIRECTORIES = {"train": "train", "valid": "valid", "test": "test"}

def load_names(data_yaml: Path) -> list[str]:
    config = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
    names = config.get("names", [])
    if isinstance(names, dict):
        names = [names[key] for key in sorted(names, key=lambda value: int(value))]
    names = [str(name) for name in names]
    if config.get("nc") != len(names):
        raise ValueError(f"data.yaml nc={config.get('nc')} but defines {len(names)} names")
    return names

def audit_dataset(dataset_root: Path, data_yaml: Path) -> tuple[list[str], dict]:
    names = load_names(data_yaml)
    class_counts = {}
    split_summaries = {}
    inventory = hashlib.sha256()

    for logical_split, directory_name in SPLIT_DIRECTORIES.items():
        images_dir = dataset_root / directory_name / "images"
        labels_dir = dataset_root / directory_name / "labels"
        images = sorted(
            p for p in images_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )
        labels = sorted(p for p in labels_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt")
        image_stems = {p.stem for p in images}
        label_stems = {p.stem for p in labels}
        missing_labels = sorted(image_stems - label_stems)
        orphan_labels = sorted(label_stems - image_stems)
        counts = Counter()
        formats = Counter()
        invalid_rows = []
        empty_label_files = 0

        for image_path in images:
            relative = image_path.relative_to(dataset_root).as_posix()
            inventory.update(f"{relative}\0{image_path.stat().st_size}\n".encode())

        for label_path in labels:
            raw = label_path.read_bytes()
            relative = label_path.relative_to(dataset_root).as_posix()
            inventory.update(relative.encode() + b"\0" + raw + b"\n")
            rows = [line.strip() for line in raw.decode("utf-8").splitlines() if line.strip()]
            if not rows:
                empty_label_files += 1
            for line_number, row in enumerate(rows, start=1):
                fields = row.split()
                try:
                    class_id = int(fields[0])
                    coordinates = [float(value) for value in fields[1:]]
                except (IndexError, ValueError):
                    invalid_rows.append(f"{relative}:{line_number}: cannot parse")
                    continue
                is_box = len(coordinates) == 4
                is_segment = len(coordinates) >= 6 and len(coordinates) % 2 == 0
                if class_id not in range(len(names)) or not (is_box or is_segment):
                    invalid_rows.append(f"{relative}:{line_number}: invalid shape or class")
                    continue
                if any(value < 0.0 or value > 1.0 for value in coordinates):
                    invalid_rows.append(f"{relative}:{line_number}: coordinate outside [0, 1]")
                    continue
                counts[class_id] += 1
                formats["box" if is_box else "segment"] += 1

        if missing_labels or orphan_labels or invalid_rows or empty_label_files:
            raise ValueError(
                f"Dataset audit failed for {logical_split}: "
                f"missing_labels={len(missing_labels)}, orphan_labels={len(orphan_labels)}, "
                f"empty_labels={empty_label_files}, invalid_rows={invalid_rows[:5]}"
            )
        class_counts[logical_split] = {name: counts[index] for index, name in enumerate(names)}
        split_summaries[logical_split] = {
            "images": len(images),
            "label_files": len(labels),
            "instances": sum(counts.values()),
            "annotation_formats": dict(formats),
        }

    summary = {
        "splits": split_summaries,
        "class_counts": class_counts,
        "total_images": sum(item["images"] for item in split_summaries.values()),
        "total_instances": sum(item["instances"] for item in split_summaries.values()),
        "inventory_sha256": inventory.hexdigest(),
        "label_normalization": LABEL_NORMALIZATION,
    }
    return names, summary

CLASS_NAMES, DATASET_AUDIT = audit_dataset(DATASET_ROOT, DATA_YAML)
print(json.dumps(DATASET_AUDIT["splits"], indent=2))
print("\nAnnotation instances per class and split:")
header = f"{'class':<14} {'train':>8} {'valid':>8} {'test':>8} {'total':>8}"
print(header)
print("-" * len(header))
for name in CLASS_NAMES:
    values = [DATASET_AUDIT["class_counts"][split][name] for split in SPLIT_DIRECTORIES]
    print(f"{name:<14} {values[0]:>8} {values[1]:>8} {values[2]:>8} {sum(values):>8}")
print(f"\nTotal images: {DATASET_AUDIT['total_images']}")
print(f"Total instances: {DATASET_AUDIT['total_instances']}")
print(f"Inventory SHA-256: {DATASET_AUDIT['inventory_sha256']}")


## 4. Seed the run and verify CUDA and dataset resolution

The seed, deterministic request, dependency versions, CUDA device, dataset audit, and actual trainer arguments are recorded in a manifest. Some GPU operations can remain nondeterministic despite these controls; the manifest makes that limitation and the exact environment inspectable.


In [ ]:
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import platform
import random
import numpy as np
import torch
import ultralytics

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available. In Colab select Runtime > Change runtime type > T4 GPU.")

CUDA_DEVICE = torch.cuda.get_device_name(0)
print(f"Python:      {platform.python_version()}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"PyTorch:     {torch.__version__}")
print(f"CUDA:       {torch.version.cuda}")
print(f"GPU:        {CUDA_DEVICE}")
print(f"Seed:       {SEED}")


In [ ]:
from ultralytics.data.utils import check_det_dataset

RESOLVED_DATA = check_det_dataset(str(DATA_YAML), autodownload=False)
for split in ("train", "val", "test"):
    resolved_path = Path(RESOLVED_DATA[split])
    if not resolved_path.is_dir():
        raise FileNotFoundError(f"Ultralytics resolved {split} to missing path: {resolved_path}")
    print(f"{split:<5} -> {resolved_path}")


## 5. Define and record the fixed training run

The cell below only defines and records the run. It refuses to reuse an existing `sitesafe_v5` directory so an earlier run cannot be mistaken for the current one. Ultralytics will also write its resolved `args.yaml` inside the run directory.


In [ ]:
from datetime import datetime, timezone

MODEL_SOURCE = "yolo11m.pt"
RUN_NAME = "sitesafe_v5"
RUNS_ROOT = AI_MODULE_ROOT / "runs" / "train"
EXPECTED_RUN_DIR = RUNS_ROOT / RUN_NAME

TRAIN_ARGS = {
    "data": str(DATA_YAML),
    "epochs": 150,
    "imgsz": 640,
    "batch": 16,
    "patience": 25,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "project": str(RUNS_ROOT),
    "name": RUN_NAME,
    "seed": SEED,
    "deterministic": True,
    "device": 0,
    "workers": 2,
    "exist_ok": False,
}

if EXPECTED_RUN_DIR.exists():
    raise FileExistsError(
        f"Run directory already exists: {EXPECTED_RUN_DIR}. Rename or move it before starting a new run."
    )

def json_safe(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return str(value)

RUN_MANIFEST = {
    "status": "configured_not_trained",
    "configured_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_source": MODEL_SOURCE,
    "requested_train_args": json_safe(TRAIN_ARGS),
    "seed": SEED,
    "environment": {
        "python": platform.python_version(),
        "ultralytics": ultralytics.__version__,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": CUDA_DEVICE,
    },
    "dataset": DATASET_AUDIT,
}
PRETRAIN_MANIFEST = WORKSPACE / "sitesafe_v5_manifest_pretrain.json"
PRETRAIN_MANIFEST.write_text(json.dumps(RUN_MANIFEST, indent=2), encoding="utf-8")
print(json.dumps({"model_source": MODEL_SOURCE, "train_args": TRAIN_ARGS}, indent=2))
print(f"Pre-training manifest: {PRETRAIN_MANIFEST}")


## 6. Train YOLO11m

Run this cell once. On a free Colab GPU, a 150-epoch medium-model run can outlast a session; early stopping may finish sooner. If Colab disconnects, preserve the complete run directory before starting over and do not treat partial weights as a completed result.


In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_SOURCE)
train_result = model.train(**TRAIN_ARGS)
RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f"Training returned without best.pt: {BEST_WEIGHTS}")

actual_args = vars(model.trainer.args)
fixed_expected = {
    "epochs": 150, "imgsz": 640, "batch": 16, "patience": 25,
    "optimizer": "AdamW", "lr0": 0.001, "seed": SEED, "deterministic": True,
}
for key, expected in fixed_expected.items():
    actual = actual_args.get(key)
    if actual != expected:
        raise AssertionError(f"Trainer argument mismatch for {key}: expected {expected!r}, got {actual!r}")

RUN_MANIFEST.update({
    "status": "training_completed_test_not_evaluated",
    "training_completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_dir": str(RUN_DIR),
    "best_weights": str(BEST_WEIGHTS),
    "actual_trainer_args": json_safe(actual_args),
})
RUN_MANIFEST_PATH = RUN_DIR / "run_manifest.json"
RUN_MANIFEST_PATH.write_text(json.dumps(RUN_MANIFEST, indent=2), encoding="utf-8")
print(f"Run directory: {RUN_DIR}")
print(f"Best weights:  {BEST_WEIGHTS}")
print(f"Run manifest:  {RUN_MANIFEST_PATH}")
print(f"Trainer args:  {RUN_DIR / 'args.yaml'}")


## 7. Mandatory held-out test evaluation

Run only after training has produced `best.pt`. This evaluates the selected checkpoint on `split="test"` at 640 pixels, prints overall precision, recall, mAP@0.5 and mAP@0.5:0.95, then prints precision, recall, mAP@0.5 and mAP@0.5:0.95 for each class. It also saves the measured results as JSON.


In [ ]:
EVAL_ARGS = {
    "data": str(DATA_YAML),
    "split": "test",
    "imgsz": 640,
    "batch": 16,
    "device": 0,
    "workers": 2,
    "seed": SEED,
    "deterministic": True,
    "plots": True,
    "project": str(AI_MODULE_ROOT / "runs" / "test"),
    "name": f"{RUN_NAME}_held_out",
    "exist_ok": False,
}

test_model = YOLO(str(BEST_WEIGHTS))
test_metrics = test_model.val(**EVAL_ARGS)
box = test_metrics.box

overall = {
    "precision": float(box.mp),
    "recall": float(box.mr),
    "map50": float(box.map50),
    "map50_95": float(box.map),
}
metric_positions = {int(class_id): position for position, class_id in enumerate(box.ap_class_index)}
per_class = []
for class_id, class_name in enumerate(CLASS_NAMES):
    if class_id not in metric_positions:
        row = {
            "class_id": class_id, "class_name": class_name,
            "test_instances": DATASET_AUDIT["class_counts"]["test"][class_name],
            "precision": None, "recall": None, "map50": None, "map50_95": None,
        }
    else:
        position = metric_positions[class_id]
        row = {
            "class_id": class_id, "class_name": class_name,
            "test_instances": DATASET_AUDIT["class_counts"]["test"][class_name],
            "precision": float(box.p[position]),
            "recall": float(box.r[position]),
            "map50": float(box.ap50[position]),
            "map50_95": float(box.ap[position]),
        }
    per_class.append(row)

print("HELD-OUT TEST METRICS (measured)")
print(f"Overall precision:    {overall['precision']:.6f}")
print(f"Overall recall:       {overall['recall']:.6f}")
print(f"Overall mAP@0.5:      {overall['map50']:.6f}")
print(f"Overall mAP@0.5:0.95: {overall['map50_95']:.6f}")
print("\nPer-class metrics:")
header = f"{'class':<14} {'instances':>10} {'precision':>11} {'recall':>11} {'mAP@0.5':>11} {'mAP@0.5:0.95':>15}"
print(header)
print("-" * len(header))
for row in per_class:
    values = [row["precision"], row["recall"], row["map50"], row["map50_95"]]
    formatted = ["n/a" if value is None else f"{value:.6f}" for value in values]
    print(
        f"{row['class_name']:<14} {row['test_instances']:>10} {formatted[0]:>11} "
        f"{formatted[1]:>11} {formatted[2]:>11} {formatted[3]:>15}"
    )

TEST_RESULTS = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "weights": str(BEST_WEIGHTS),
    "split": "test",
    "evaluation_args": json_safe(EVAL_ARGS),
    "overall": overall,
    "per_class": per_class,
}
TEST_RESULTS_PATH = RUN_DIR / "held_out_test_metrics.json"
TEST_RESULTS_PATH.write_text(json.dumps(TEST_RESULTS, indent=2), encoding="utf-8")
RUN_MANIFEST.update({
    "status": "training_and_held_out_test_completed",
    "held_out_test": TEST_RESULTS,
})
RUN_MANIFEST_PATH.write_text(json.dumps(RUN_MANIFEST, indent=2), encoding="utf-8")
print(f"\nSaved test metrics: {TEST_RESULTS_PATH}")


## 8. Export `best.pt` and the evidence files

This copies the selected weights, manifest, trainer arguments, and held-out metrics into one artifact directory and prints the weights checksum. Optionally copy that directory to Drive or download `best.pt`. Do not replace repository weights or update the report until the held-out metrics above have actually been reviewed.


In [ ]:
EXPORT_DIR = WORKSPACE / "artifacts" / RUN_NAME
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORTED_BEST = EXPORT_DIR / "best.pt"
shutil.copy2(BEST_WEIGHTS, EXPORTED_BEST)
shutil.copy2(RUN_MANIFEST_PATH, EXPORT_DIR / RUN_MANIFEST_PATH.name)
shutil.copy2(TEST_RESULTS_PATH, EXPORT_DIR / TEST_RESULTS_PATH.name)
shutil.copy2(DATA_YAML, EXPORT_DIR / "data.yaml")
TRAINER_ARGS_PATH = RUN_DIR / "args.yaml"
if TRAINER_ARGS_PATH.is_file():
    shutil.copy2(TRAINER_ARGS_PATH, EXPORT_DIR / TRAINER_ARGS_PATH.name)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print(f"Artifact directory: {EXPORT_DIR}")
print(f"Exported best.pt:  {EXPORTED_BEST}")
print(f"best.pt SHA-256:   {sha256_file(EXPORTED_BEST)}")

COPY_EXPORT_TO_DRIVE = False  # Set True after reviewing the destination below.
DRIVE_EXPORT_DIR = Path("/content/drive/MyDrive/site-safe/model-exports/sitesafe_v5")
if COPY_EXPORT_TO_DRIVE:
    if not MOUNT_GOOGLE_DRIVE:
        raise RuntimeError("Mount Google Drive before enabling COPY_EXPORT_TO_DRIVE")
    if DRIVE_EXPORT_DIR.exists():
        raise FileExistsError(f"Drive export already exists: {DRIVE_EXPORT_DIR}")
    shutil.copytree(EXPORT_DIR, DRIVE_EXPORT_DIR)
    print(f"Copied evidence bundle to: {DRIVE_EXPORT_DIR}")

# Optional browser download:
# from google.colab import files
# files.download(str(EXPORTED_BEST))
